In [ ]:
import numpy as np
from scipy.ndimage import convolve


def gaussian_stack(img, num_levels, initial_sigma=1.0, sigma_multiplier=2.0):
    """
    Create a Gaussian stack - each level is filtered but NOT downsampled
    """
    stack = []
    current = img.astype(np.float64)
    stack.append(current)  # Level 0 is original image
    
    sigma = initial_sigma
    
    for i in range(1, num_levels):
        # Create Gaussian kernel for this level
        kernel = gaussian_filter(sigma)
        
        # Apply Gaussian filter (no downsampling)
        if len(img.shape) == 3:  # Color image
            filtered = np.zeros_like(current)
            for channel in range(img.shape[2]):
                filtered[:, :, channel] = convolve(current[:, :, channel], kernel, mode='reflect')
        else:  # Grayscale image
            filtered = convolve(current, kernel, mode='reflect')
        
        stack.append(filtered)
        current = filtered  # Use filtered image for next level
        sigma *= sigma_multiplier  # Increase sigma for next level
    
    return stack

def laplacian_stack2(img, num_levels, initial_sigma=1.0, sigma_multiplier=2.0):
    """
    Create a Laplacian stack - difference between successive Gaussian levels
    """
    # First create Gaussian stack
    gaussian_stack_list = gaussian_stack(img, num_levels, initial_sigma, sigma_multiplier)
    
    laplacian_stack_list = []
    
    # Laplacian levels are differences between Gaussian levels
    for i in range(len(gaussian_stack_list) - 1):
        # Laplacian = current Gaussian level - next Gaussian level
        lap_level = gaussian_stack_list[i] - gaussian_stack_list[i + 1]
        laplacian_stack_list.append(lap_level)
    
    # Last level is the final Gaussian level
    laplacian_stack_list.append(gaussian_stack_list[-1])
    
    return laplacian_stack_list, gaussian_stack_list

def display_stacks2(gaussian_stack, laplacian_stack, title="Gaussian and Laplacian Stacks"):
    """Display the stacks for visualization"""
    num_levels = len(gaussian_stack)
    
    fig, axes = plt.subplots(2, num_levels, figsize=(4*num_levels, 8))
    
    for i in range(num_levels):
        # Display Gaussian stack level
        if len(gaussian_stack[i].shape) == 2:  # Grayscale
            axes[0, i].imshow(gaussian_stack[i], cmap='gray')
        else:  # Color
            axes[0, i].imshow(np.clip(gaussian_stack[i], 0, 1))
        axes[0, i].set_title(f'Gaussian Level {i}')
        axes[0, i].axis('off')
        
        # Display Laplacian stack level (normalized for visualization)
        if i < len(laplacian_stack):
            lap_display = laplacian_stack[i]
            if len(lap_display.shape) == 2:  # Grayscale
                # Normalize for better visualization
                if lap_display.max() > lap_display.min():
                    lap_display = (lap_display - lap_display.min()) / (lap_display.max() - lap_display.min())
                axes[1, i].imshow(lap_display, cmap='gray')
            else:  # Color
                axes[1, i].imshow(np.clip(lap_display + 0.5, 0, 1))  # Center around 0.5
            axes[1, i].set_title(f'Laplacian Level {i}')
            axes[1, i].axis('off')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
img = plt.imread('../data/apple.jpeg') / 255.0

l, g = laplacian_stack2(img, 5, 3.0, 2.0)
display_stacks2(g, l)